# Data gathering

- Get documents with ids
- Get ground truth data
- Evaluate

Task:
https://courses.datatalks.club/llm-zoomcamp-2025/homework/hw3

In [2]:
import requests
import pandas as pd
from tqdm.auto import tqdm


url_prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/'
docs_url = url_prefix + 'search_evaluation/documents-with-ids.json'
documents = requests.get(docs_url).json()

ground_truth_url = url_prefix + 'search_evaluation/ground-truth-data.csv'
df_ground_truth = pd.read_csv(ground_truth_url)
# df_ground_truth = df_ground_truth[df_ground_truth.course == 'machine-learning-zoomcamp']
ground_truth = df_ground_truth.to_dict(orient='records')

Evalution metrics

- Hit rate (recall) measures the success of a retrieval system by indicating how often a relevant item appears within the top-N recommendations

- MRR (Mean reciprocal rank) evaluate the performance of information retrieval and recom. systems, particularly in the context of ranking results.

In [3]:
def hit_rate(data):
    count = 0
    for line in data:
        if True in line:
            count += 1
    return count / len(data)

In [4]:
def mrr(data):
    total_score = 0
    for line in data:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)
    return total_score / len(data)

In [9]:
def evaluate(ground_truth, search_function):
    relevance_total = []
    
    for q in tqdm(ground_truth):
        doc_id = q["document"]
        results = search_function(q['question'], q['course'])
        relevance = [doc["id"] == doc_id for doc in results]
        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

### Question 1. 
Hitrate for minsearch text (1 point)

In [10]:
import minsearch

index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course", "id"]
)

index.fit(documents)

In [11]:
query = "I just discovered the course. Can I still join?"

def search(query, course):
    boost = {'question': 1.5, 'section': 0.1}

    results = index.search(
        query=query,
        filter_dict={'course': course},
        boost_dict=boost,
        num_results=5
    )

    return results

In [12]:
evaluate(ground_truth, search)

  0%|          | 0/4627 [00:00<?, ?it/s]

{'hit_rate': 0.848714069591528, 'mrr': 0.7288235717887772}

### Question 2. 
MRR Vector search (question field) (1 point)

In [13]:
from minsearch import VectorSearch

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline

In [14]:
texts = []
for doc in documents:
    t = doc["question"]
    texts.append(t)

pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
X = pipeline.fit_transform(texts)

In [15]:
vindex = VectorSearch(keyword_fields={"course"})
vindex.fit(X, documents)

In [27]:
def minsearch_vector_search(vector, course):
    return vindex.search(
        vector,
        filter_dict={'course': course},
        num_results=5
    )

def question_text_vector(question, course):
    question = question
    course = course

    v_q = pipeline.transform([question])

    return minsearch_vector_search(v_q, course)

In [28]:
evaluate(ground_truth, question_text_vector)

  0%|          | 0/4627 [00:00<?, ?it/s]

{'hit_rate': 0.48195374972984656, 'mrr': 0.3573085512571141}

### Question 3. 
Hitrate Vector search (question + text fields)

In [29]:
texts = []
for doc in documents:
    text = doc["question"] + " " + doc["text"]
    texts.append(text)

pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
X = pipeline.fit_transform(texts)

In [30]:
vindex = VectorSearch(keyword_fields={"course"})
vindex.fit(X, documents)

In [31]:
def minsearch_vector_search(vector, course):
    return vindex.search(
        vector,
        filter_dict={'course': course},
        num_results=5
    )

def question_text_vector(question, course):
    question = question
    course = course

    v_q = pipeline.transform([question])

    return minsearch_vector_search(v_q, course)

In [32]:
evaluate(ground_truth, question_text_vector)

  0%|          | 0/4627 [00:00<?, ?it/s]

{'hit_rate': 0.8210503566025502, 'mrr': 0.6717347453353508}

### Question 4. 
MRR Qdrant (1 point)

```bash
1. docker pull qdrant/qdrant

2. docker run -p 6333:6333 -p 6334:6334
-v "$(pwd)/qdrant_storage:/qdrant/storage:z"
qdrant/qdrant
```

In [33]:
!docker run -d -p 6333:6333 -p 6334:6334 \
   -v "./qdrant_storage:/qdrant/storage:z" \
   qdrant/qdrant

a1b559714c053d49cbf5eabbdfaa9961e5b030f1934c8ef30b6a0e1bc2c560a5


In [34]:
!python -m pip install -q "qdrant-client[fastembed]>=1.14.2"

In [36]:
from qdrant_client import QdrantClient

client = QdrantClient("http://localhost:6333")
client.get_collections()

CollectionsResponse(collections=[])

In [37]:
text = doc['question'] + ' ' + doc['text']
model_handle = "jinaai/jina-embeddings-v2-small-en"
limit = 5

In [42]:
from qdrant_client import models

collection_name = "zoomcamp-rag_1"
EMBEDDING_DIMENSIONALITY = 512

if collection_name in [c.name for c in client.get_collections().collections]:
    client.delete_collection(collection_name=collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIMENSIONALITY,
        distance=models.Distance.COSINE
    )
)

True

In [41]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp',
 'id': 'c02e79ef'}

In [43]:
points = []
id = 0

for doc in documents:
    point = models.PointStruct(
        id=id,
        vector=models.Document(text=doc['question'] + ' ' + doc['text'], model=model_handle), #embed text locally with "jinaai/jina-embeddings-v2-small-en" from FastEmbed
        payload={
            "text": doc['text'],
            "section": doc['section'],
            "course": doc['course'],
            "id": doc['id']
        } #save all needed metadata fields
    )
    points.append(point)

    id += 1

In [45]:
client.upsert(
    collection_name=collection_name,
    points=points
)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

onnx/model.onnx:   0%|          | 0.00/130M [00:00<?, ?B/s]

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [46]:
def search(query: str, limit: int = 5) -> list[models.ScoredPoint]:

    results = client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model=model_handle 
        ),
        limit=limit,
        with_payload=True
    )

    return results

In [47]:
def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q['question'], q['course'])
        relevance = [point.payload['id'] == doc_id for point in results.points]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

In [ ]:
client.create_payload_index(
    collection_name=collection_name,
    field_name="course",
    field_schema="keyword"
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [49]:
def search_in_course(query: str, course="mlops-zoomcamp", limit: int = 5) -> list[models.ScoredPoint]:
    results = client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model=model_handle
        ),
        query_filter=models.Filter( # filter by course name
            must=[
                models.FieldCondition(
                    key="course",
                    match=models.MatchValue(value=course)
                )
            ]
        ),
        limit=limit, # top closest matches
        with_payload=True #to get metadata in the results
    )

    return results

In [50]:
evaluate(ground_truth, search_in_course)

  0%|          | 0/4627 [00:00<?, ?it/s]

{'hit_rate': 0.9299762264966501, 'mrr': 0.8517722066133576}

### Question 5. 
Average cosine (1 point)

In [52]:
import pandas as pd

url_prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/rag_evaluation/data/'
results_url = url_prefix + 'results-gpt4o-mini.csv'

df_results = pd.read_csv(results_url)

In [53]:
import numpy as np

def cosine(u, v):
    u_norm = np.sqrt(u.dot(u))
    v_norm = np.sqrt(v.dot(v))
    return u.dot(v) / (u_norm * v_norm)

In [54]:
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

In [55]:
pipeline.fit(df_results.answer_llm + ' ' + df_results.answer_orig + ' ' + df_results.question)


Pipeline(steps=[('tfidfvectorizer', TfidfVectorizer(min_df=3)),
                ('truncatedsvd',
                 TruncatedSVD(n_components=128, random_state=1))])

In [56]:
v_llm = pipeline.transform(df_results.answer_llm)
v_orig = pipeline.transform(df_results.answer_orig)

In [57]:
result = []
for i in range(0, len(v_llm)):
    result.append(cosine(v_llm[i], v_orig[i]))

mean = sum(result) / len(result)

In [58]:
mean

0.8415841233490399

### Question 6. 
Average Rouge-1 F1 (1 point)

ROUGE-1 (F1) is a metric used to evaluate the quality of text summaries, specifically focusing on the overlap of individual words (unigrams) between a generated summary and a reference (human-written) summary.

- ROUGE-1:
Focuses on unigram overlap, meaning it counts how many individual words from the reference summary are also present in the generated summary. 

- Recall:
Measures how much of the reference summary is captured in the generated summary. It's calculated by dividing the number of matching words by the total number of words in the reference summary. 

- Precision:
Measures how much of the generated summary is relevant to the reference summary. It's calculated by dividing the number of matching words by the total number of words in the generated summary. 

- F1-score:
The harmonic mean of recall and precision, providing a balanced score that considers both how much of the reference is captured and how much of the generated is relevant. 

In [59]:
!pip install rouge

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [60]:
from rouge import Rouge
rouge_scorer = Rouge()

r = df_results.iloc[10]
scores = rouge_scorer.get_scores(r.answer_llm, r.answer_orig)[0]
scores

{'rouge-1': {'r': 0.45454545454545453,
  'p': 0.45454545454545453,
  'f': 0.45454544954545456},
 'rouge-2': {'r': 0.21621621621621623,
  'p': 0.21621621621621623,
  'f': 0.21621621121621637},
 'rouge-l': {'r': 0.3939393939393939,
  'p': 0.3939393939393939,
  'f': 0.393939388939394}}

In [61]:
rouge1 = []
for i in range(len(df_results)):
    doc = df_results.iloc[i]
    score = rouge_scorer.get_scores(doc.answer_llm, doc.answer_orig)[0]
    rouge1.append(score['rouge-1']['f'])

mean = sum(rouge1) / len(rouge1)
mean

0.3516946452113944